## Analysing NHS staff survey responses

In [3]:
# Modify pd.read_csv() to read in and modify survey data
import pandas as pd

class ReadModifyCSV(pd.DataFrame):
    @classmethod
    def read_csv(cls, *args, **kwargs):
        temp = pd.read_csv(*args, **kwargs)
        
        temp = temp.drop([0, 1], axis='index') # Remove first two rows
        temp = temp.reset_index(drop=True) # Refresh index
        temp = temp.iloc[:, [1]] # Keep only second column (`[]` keeps it as df, not series)
        temp.columns = ['comment'] # Name the single column
        temp.insert(0, 'id', range(1, len(temp) + 1)) # Create id column

        # Return an instance of the custom class
        return cls(temp)
    
    def __init__(self, data, *args, **kwargs):
        super().__init__(data, *args, **kwargs)

In [4]:
df = ReadModifyCSV.read_csv('data/community_care.csv', encoding='cp1252')
df.head()

,id,comment
0,1,I have retired and returned. My dissatisfact...
1,2,A lot of our IT equipment is coming to the end...
2,3,A number of clinical areas are short staffed a...
3,4,Absolutely disgusted with the way I have just ...
4,5,An expanding department with lots of new devel...


In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import ollama

# Activate OpenAI API Key
load_dotenv('api.env')
openai_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openai_api_key)

In [6]:
SplittingPrompt = f"""
Your task is to split an open-text survey response from an NHS staff survey into segments by inserting new lines (`\\n`) between distinct topics. Follow these rules strictly:

1. **Preserve Originality**: Do not rephrase, summarise, or alter the wording of the response in any way. Copy the text exactly as it appears, adding only `\\n` characters where necessary.
2. **Split by Topic, Not Just Sentences**: Insert a `\\n` character **only** when the respondent clearly transitions to a **new, distinct topic**. **Sentences discussing different aspects of the same issue should not be split.**
3. **Avoid Over-Splitting**: A topic **is not just a sentence–it is a concept.** Do not separate sentences just because they present different details. **If two sentences naturally flow together, they should remain together.**
4. **No Commentary**: Do not add headings, summaries, or analysis. Your output should be the original text with `\\n` characters added where appropriate.

---

### **Example 1: Correct Splitting**
--Original:  
"I feel unsupported by management. IT issues also delay my work. My colleagues are very helpful."

--Output:  
"I feel unsupported by management.  

IT issues also delay my work.  

My colleagues are very helpful."

---

### **Example 2: A Response That Should Not Be Split Further**
--Original:  
"The workload has been intense, and deadlines keep piling up. It feels like management doesn’t always see how much we’re juggling, which adds to the pressure. My team, though, has been fantastic–we support each other and keep things running even when it’s stressful."

--Output:  
"The workload has been intense, and deadlines keep piling up. It feels like management doesn’t always see how much we’re juggling, which adds to the pressure.  

My team, though, has been fantastic–we support each other and keep things running even when it’s stressful."

---

### **Example 3: A Response That Should NOT Be Split at All**
--Original:  
"A number of clinical areas are short staffed and unable to recruit. The pressure placed on current staff can be unrealistic and makes the job exhausting and emotionally draining."

--Incorrect Output (Do not do this):  
"A number of clinical areas are short staffed and unable to recruit.  

The pressure placed on current staff can be unrealistic and makes the job exhausting and emotionally draining."

--Correct Output:  
"A number of clinical areas are short staffed and unable to recruit. The pressure placed on current staff can be unrealistic and makes the job exhausting and emotionally draining."

---

### **Example 4: When to Split and When Not To**
--Original:  
"Our department has moved three times in the past two years. It has been incredibly disruptive, and morale is low. Staff turnover has increased because people are frustrated with the constant changes.  

The new hybrid working policy is also causing problems. Some roles are fully remote while others are required to be on-site, but there’s no clear reasoning for why. This inconsistency is frustrating."

--Output:  
"Our department has moved three times in the past two years. It has been incredibly disruptive, and morale is low. Staff turnover has increased because people are frustrated with the constant changes.  

The new hybrid working policy is also causing problems. Some roles are fully remote while others are required to be on-site, but there’s no clear reasoning for why. This inconsistency is frustrating."

---

### **Final Instructions**
Now, apply these rules to the following response:

"""

In [7]:
responses_split = []

# Iterate over data frame
def GetResponses(df, TextCol, Prompt, Print=False, approach='openai'):
    responses = []
    
    for index, row in df.iterrows():
        Content = row[TextCol]
        
        # Print processed text if Print=True
        if Print:
            print(f"***\n\n--Original Text:\n {Content}")
        
        # Create a dynamic prompt which combines static prompt with content from df
        dynamic_prompt = Prompt + Content
        #print(dynamic_prompt)  # Uncomment for prompt debugging only
        
        
        if approach == 'openai':
            try:
                completion = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {"role": "system", "content": "You are a helpful assistant."},
                        {"role": "user", "content": dynamic_prompt}
                    ], temperature=0
                )
                response = completion.choices[0].message.content
                
            except Exception as e:
                print(f"Error at index {index}: {e}")
                response = None
        
        elif approach == 'ollama':
            try:
                response = ollama.chat(model="gemma2:27b", messages=[
                    {"role": "user", "content": dynamic_prompt}
                ])
                response = response['message']['content']
            
            except Exception as e:
                print(f"Error at index {index}: {e}")
                response = None
        
        if response:
            if Print:
                print(f"\n\n--Response:\n{response}")
            # Store responses
            responses.append(response)
        
    return responses

In [8]:
# Sample first 10 rows
sample = df.iloc[0:10]
sample

,id,comment
0,1,I have retired and returned. My dissatisfact...
1,2,A lot of our IT equipment is coming to the end...
2,3,A number of clinical areas are short staffed a...
3,4,Absolutely disgusted with the way I have just ...
4,5,An expanding department with lots of new devel...
5,6,As a [job title removed] I have never felt so ...
6,7,"as above in your list of AHP Occupations, Podi..."
7,8,As usual there is no option for phlebotomy in ...
8,9,Before semi retirement due to underlying progr...
9,10,Being a new comer am facing language problems....


In [9]:
# Test our prompt on our sample & check output
sample['split_comment'] = GetResponses(sample, 'comment', Prompt=SplittingPrompt, Print=True, approach='openai')

***

--Original Text:
  I have retired and returned.  My dissatisfaction with the organisation stems from the estates policy which appears to be all about saving money.  Our Department has been moved 3 times in the past 2 years and now our base is outside of the geographical area we serve as the  trust has become bigger after taking over 2 other trusts.  The NHS appear to have use the pandemic to sell off all its estates.  Patients miss out as services are no longer local to them.


--Response:
I have retired and returned.  

My dissatisfaction with the organisation stems from the estates policy which appears to be all about saving money.  

Our Department has been moved 3 times in the past 2 years and now our base is outside of the geographical area we serve as the trust has become bigger after taking over 2 other trusts.  

The NHS appear to have used the pandemic to sell off all its estates.  

Patients miss out as services are no longer local to them.
***

--Original Text:
 A lot o

/tmp/ipykernel_1546600/1930646819.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample['split_comment'] = GetResponses(sample, 'comment', Prompt=SplittingPrompt, Print=True, approach='openai')


In [10]:
sample

,id,comment,split_comment
0,1,I have retired and returned. My dissatisfact...,I have retired and returned. \n\nMy dissatisf...
1,2,A lot of our IT equipment is coming to the end...,A lot of our IT equipment is coming to the end...
2,3,A number of clinical areas are short staffed a...,"""A number of clinical areas are short staffed ..."
3,4,Absolutely disgusted with the way I have just ...,Absolutely disgusted with the way I have just ...
4,5,An expanding department with lots of new devel...,An expanding department with lots of new devel...
5,6,As a [job title removed] I have never felt so ...,As a [job title removed] I have never felt so ...
6,7,"as above in your list of AHP Occupations, Podi...","as above in your list of AHP Occupations, Podi..."
7,8,As usual there is no option for phlebotomy in ...,As usual there is no option for phlebotomy in ...
8,9,Before semi retirement due to underlying progr...,Before semi retirement due to underlying progr...
9,10,Being a new comer am facing language problems....,Being a new comer am facing language problems....


In [11]:
# Run function on entire df
df['split_comment'] = GetResponses(df, 'comment', Prompt=SplittingPrompt, Print=False, approach='openai')


In [12]:
# Split by double new-line characters (`\n\n`)

def split_comments(df, split_col="split_comment"):
    
    new_rows = []  # Create empty list to store transformed data

    for _, row in df.iterrows():  
        # `iterrows()` gives us both the index and the row. We ignore the index by using `_`
        
        # Check if the value in the specified column (split_col) is not empty or NaN
        if pd.notna(row[split_col]):  
            # Split the text based on double newline characters (`\n\n`), which separate different segments
            comment_splits = row[split_col].split("\n\n")  

            # Iterate over each split segment
            for split in comment_splits:
                # Strip leading/trailing whitespace to clean up the text
                split = split.strip()

                # Only add non-empty strings (to prevent blank rows in the output)
                if split:
                    new_rows.append({'id': row['id'], 'comment': split})
    
    # Convert the list of new rows into a DataFrame and return it
    return pd.DataFrame(new_rows)

split_df = split_comments(df, "split_comment")
split_sample = split_comments(sample, "split_comment")
split_df


,id,comment
0,1,I have retired and returned.
1,1,My dissatisfaction with the organisation stems...
2,1,Our Department has been moved 3 times in the p...
3,1,The NHS appear to have used the pandemic to se...
4,1,Patients miss out as services are no longer lo...
...,...,...
566,258,please put Podiatry on your list
567,259,"""Why do we waste so much money and paper and s..."
568,260,"""Would like more apprenticeship opportunities."""
569,261,"""Would not recommend healthcare as a career to..."


In [13]:
LabellingPrompt = f"""

You will be given a brief snippet of text from an NHS staff survey. Your goal is to return both a theme from a given list, and a sentiment (either: positive, negative, or neutral).

Below is the list of themes respective subthemes.

**Estates**
- Condition of facilities
- Office/location moves
- Space constraints
- Other

**Staffing**
- Understaffing
- Turnover and understaffing
- Recruitment
- Other

**Workload**
- Caseloads
- Performance targets/KPIs
- Demand
- Other

**Management**
- Visibility and approachability of managers
- Management style
- Consultation in decision-making
- Other

**Comms**
- Internal communication
- Cross-team/service
- Connection between senior leadership and frontline staff
- Other

**Culture**
- Bullying, harassment, or intimidation
- Nepotism/favouritism in promotions
- Blame culture
- Other

**Wellbeing**
- Stress, anxiety, burnout
- Access to support
- Post-COVID fatigue and residual pressures
- Other

**Pay**
- Banding mismatches / under-banding
- Expenses and mileage
- Cost-of-living
- Other

**Progression**
- Training
- Career development
- Other

**Flexibility**
- Remote vs. onsite 
- Desk-hopping or hot-desking
- Work-life balance
- Other

**IT**
- Outdated IT
- IT not working
- Overly complex digital systems (e.g. duplicative data entry)
- IT support
- Other

**EDI and fairness**
- Underrepresented professions
- Cultural/language barriers
- EDI policies / teams
- Other

**Admin**
- Excessive or complex documentation
- Training overload
- Unresponsive HR or recruitment processes
- Other

**Mergers**
- Disconnection post-acquisition (e.g. staff feeling “lost”)
- Separate or conflicting systems
- Multiple reorganisations
- Other

**Patients**
- Patient care
- Complexity of patient needs
- Access issues
- Other

**Recognition**
- Acknowledgment of effort
- Other

**No theme** (When no other theme is suitable)
- N/A

**

Please format your response as follows: [theme//subtheme//sentiment] -- for example: Wellbeing//Access to support//Positive

Do not add headings, summaries, or any analysis. Your response should be as outlined above **only**.

"""

In [14]:
GetResponses(split_sample, 'comment', LabellingPrompt, Print=True, approach='openai')

***

--Original Text:
 I have retired and returned.


--Response:
No theme//N/A//Neutral
***

--Original Text:
 My dissatisfaction with the organisation stems from the estates policy which appears to be all about saving money.


--Response:
Estates//Other//Negative
***

--Original Text:
 Our Department has been moved 3 times in the past 2 years and now our base is outside of the geographical area we serve as the trust has become bigger after taking over 2 other trusts.


--Response:
Estates//Office/location moves//Negative
***

--Original Text:
 The NHS appear to have used the pandemic to sell off all its estates.


--Response:
Estates//Other//Negative
***

--Original Text:
 Patients miss out as services are no longer local to them.


--Response:
Patients//Access issues//Negative
***

--Original Text:
 A lot of our IT equipment is coming to the end of it's life and our departmental budgets are feeling the strain.


--Response:
IT//Outdated IT//Negative
***

--Original Text:
 Our budget

['No theme//N/A//Neutral',
 'Estates//Other//Negative',
 'Estates//Office/location moves//Negative',
 'Estates//Other//Negative',
 'Patients//Access issues//Negative',
 'IT//Outdated IT//Negative',
 'Estates//Office/location moves//Negative',
 'Pay//Expenses and mileage//Negative',
 'IT//Overly complex digital systems//Negative',
 'Progression//Career development//Positive',
 'Culture//Bullying, harassment, or intimidation//Negative',
 'No theme//N/A//Neutral',
 'Staffing//Understaffing//Negative',
 'Culture//Bullying, harassment, or intimidation//Negative',
 'Culture//Bullying, harassment, or intimidation//Negative',
 'No theme//N/A//Negative',
 'Wellbeing//Stress, anxiety, burnout//Negative',
 'Management//Consultation in decision-making//Negative',
 'Culture//Blame culture//Negative',
 'EDI and fairness//Underrepresented professions//Negative',
 'Progression//Career development//Negative',
 'IT//Outdated IT//Negative',
 'No theme//N/A//Negative',
 'Recognition//Acknowledgment of eff

In [15]:
split_df['assignment'] = GetResponses(split_df, 'comment', LabellingPrompt, Print=False, approach='openai')

In [16]:
split_df

,id,comment,assignment
0,1,I have retired and returned.,No theme//N/A//Neutral
1,1,My dissatisfaction with the organisation stems...,Estates//Other//Negative
2,1,Our Department has been moved 3 times in the p...,Estates//Office/location moves//Negative
3,1,The NHS appear to have used the pandemic to se...,Estates//Other//Negative
4,1,Patients miss out as services are no longer lo...,Patients//Access issues//Negative
...,...,...,...
566,258,please put Podiatry on your list,No theme//N/A//Neutral
567,259,"""Why do we waste so much money and paper and s...",Admin//Excessive or complex documentation//Neg...
568,260,"""Would like more apprenticeship opportunities.""",Progression//Training//Positive
569,261,"""Would not recommend healthcare as a career to...",Culture//Blame culture//Negative


In [17]:
# Split the 'assignment' column and check the length of the split result
split_results = split_df['assignment'].str.split('//')

# Flag rows where the split result does not have exactly three parts
split_df['split_length'] = split_results.apply(len)
invalid_rows = split_df[split_df['split_length'] != 3]

# Print or handle invalid rows in a readable format
print("Invalid rows:")
print(invalid_rows.to_string(index=False))

# Proceed with the split operation only for valid rows
valid_split_df = split_df[split_df['split_length'] == 3]
valid_split_df[['theme', 'subtheme', 'sentiment']] = pd.DataFrame(valid_split_df['assignment'].str.split('//').tolist(), index=valid_split_df.index)

# Drop the 'split_length' and assignment cols
valid_split_df = valid_split_df.drop(columns=['split_length', 'assignment'])

Invalid rows:
 id                                                                                                                                                                                                                                                                                                                                                                                                                                                   comment                                                                                                                                        assignment  split_length
151 My manager is amazing and so supportive. However, the stress they are under and the rest of the team is not good. Amazing staff members are going elsewhere for work because of how heavy the workload is. None of my colleagues take a break during their shift. They have to decide whether they take a break and finish late or don't take a break and finish on time. The pressure p

/tmp/ipykernel_1546600/3497616343.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_split_df[['theme', 'subtheme', 'sentiment']] = pd.DataFrame(valid_split_df['assignment'].str.split('//').tolist(), index=valid_split_df.index)
/tmp/ipykernel_1546600/3497616343.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_split_df[['theme', 'subtheme', 'sentiment']] = pd.DataFrame(valid_split_df['assignment'].str.split('//').tolist(), index=valid_split_df.index)
/tmp/ipykernel_1546600/3497616343.py:14:

In [18]:
valid_split_df

,id,comment,theme,subtheme,sentiment
0,1,I have retired and returned.,No theme,N/A,Neutral
1,1,My dissatisfaction with the organisation stems...,Estates,Other,Negative
2,1,Our Department has been moved 3 times in the p...,Estates,Office/location moves,Negative
3,1,The NHS appear to have used the pandemic to se...,Estates,Other,Negative
4,1,Patients miss out as services are no longer lo...,Patients,Access issues,Negative
...,...,...,...,...,...
566,258,please put Podiatry on your list,No theme,N/A,Neutral
567,259,"""Why do we waste so much money and paper and s...",Admin,Excessive or complex documentation,Negative
568,260,"""Would like more apprenticeship opportunities.""",Progression,Training,Positive
569,261,"""Would not recommend healthcare as a career to...",Culture,Blame culture,Negative


In [19]:
import pandas as pd

# Total count of comments
total_count = len(valid_split_df)

# Group by 'theme' and calculate counts and percentages
theme_table = valid_split_df.groupby("theme")["id"].count().reset_index()
theme_table.columns = ["Theme", "Count"]
theme_table["Percentage"] = (theme_table["Count"] / total_count * 100).round(1)

# Group by 'theme' and 'subtheme' and calculate counts and percentages
subtheme_table = valid_split_df.groupby(["theme", "subtheme"])["id"].count().reset_index()
subtheme_table.columns = ["Theme", "Subtheme", "Count"]
subtheme_table["Percentage"] = (subtheme_table["Count"] / total_count * 100).round(1)

# Group by 'sentiment' and calculate counts and percentages
sentiment_table = valid_split_df.groupby("sentiment")["id"].count().reset_index()
sentiment_table.columns = ["Sentiment", "Count"]
sentiment_table["Percentage"] = (sentiment_table["Count"] / total_count * 100).round(1)

# Group by 'theme' and 'sentiment' to create a new table
theme_sentiment_table = valid_split_df.groupby(["theme", "sentiment"])["id"].count().reset_index()
theme_sentiment_table.columns = ["Theme", "Sentiment", "Count"]
theme_sentiment_table["Percentage"] = (theme_sentiment_table["Count"] / total_count * 100).round(1)

# Convert to markdown format
theme_markdown = theme_table.to_markdown(index=False)
subtheme_markdown = subtheme_table.to_markdown(index=False)
sentiment_markdown = sentiment_table.to_markdown(index=False)
theme_sentiment_markdown = theme_sentiment_table.to_markdown(index=False)

# Print markdown code blocks
print("# Themes Count and Percentage")
print(theme_markdown)
print("\n\n")

print("# Themes and Subthemes Count and Percentage")
print(subtheme_markdown)
print("\n\n")

print("# Sentiment Count and Percentage")
print(sentiment_markdown)
print("\n\n")

print("# Themes by Sentiment Count and Percentage")
print(theme_sentiment_markdown)
print("\n\n")


# Themes Count and Percentage
| Theme            |   Count |   Percentage |
|:-----------------|--------:|-------------:|
| Admin            |      22 |          3.9 |
| Comms            |      22 |          3.9 |
| Culture          |      76 |         13.3 |
| EDI and fairness |      13 |          2.3 |
| Estates          |      17 |          3   |
| Flexibility      |      20 |          3.5 |
| IT               |      19 |          3.3 |
| Management       |      72 |         12.6 |
| Mergers          |       9 |          1.6 |
| No theme         |      51 |          8.9 |
| Patients         |      18 |          3.2 |
| Pay              |      34 |          6   |
| Progression      |      31 |          5.4 |
| Recognition      |      11 |          1.9 |
| Staffing         |      59 |         10.4 |
| Training         |       1 |          0.2 |
| Wellbeing        |      52 |          9.1 |
| Workload         |      43 |          7.5 |



# Themes and Subthemes Count and Percentage
| T